# Data Cleaning and Feature Engineering

## Objective

This notebook transforms the audited raw datasets into analysis-ready data while preserving the original source files.

The notebook will:

- Create independent working copies of the raw datasets
- Standardize text, date, and currency fields
- Create validated analytical order and customer keys
- Calculate shipping duration and profitability metrics
- Standardize return-region labels and add an order-level return flag
- Preserve documented data limitations
- Validate the cleaned datasets
- Export reproducible processed CSV files

## 1. Environment and Project Paths

I import the required libraries and locate the raw and processed data folders without hard-coding a personal computer path.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

current_folder = Path.cwd()

if (current_folder / "data" / "raw").exists():
    project_root = current_folder
elif (current_folder.parent / "data" / "raw").exists():
    project_root = current_folder.parent
else:
    raise FileNotFoundError(
        "Project root containing data/raw was not found."
    )

raw_data_folder = project_root / "data" / "raw"
processed_data_folder = project_root / "data" / "processed"

processed_data_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Current folder:", current_folder.name)
print("Project folder:", project_root.name)
print(
    "Raw data folder:",
    raw_data_folder.relative_to(project_root)
)
print(
    "Processed data folder:",
    processed_data_folder.relative_to(project_root)
)
print("Raw folder exists:", raw_data_folder.exists())
print(
    "Processed folder exists:",
    processed_data_folder.exists()
)

Current folder: notebooks
Project folder: ecommerce_sales_customer_profitability_analytics
Raw data folder: data\raw
Processed data folder: data\processed
Raw folder exists: True
Processed folder exists: True


## 2. Load Raw Data and Create Working Copies

I load the source CSV files into raw DataFrames and create separate working copies for cleaning and feature engineering.

In [2]:
orders_path = raw_data_folder / "orders_data.csv"
returns_path = raw_data_folder / "returns_data.csv"
people_path = raw_data_folder / "people_data.csv"

orders_raw = pd.read_csv(
    orders_path,
    encoding="utf-8-sig",
    dtype={"Postal Code": "string"}
)

returns_raw = pd.read_csv(
    returns_path,
    encoding="utf-8-sig"
)

people_raw = pd.read_csv(
    people_path,
    encoding="utf-8-sig"
)

orders_clean = orders_raw.copy()
returns_clean = returns_raw.copy()
people_clean = people_raw.copy()

loading_summary = pd.DataFrame({
    "Dataset": ["Orders", "Returns", "People"],
    "Rows": [
        orders_clean.shape[0],
        returns_clean.shape[0],
        people_clean.shape[0]
    ],
    "Columns": [
        orders_clean.shape[1],
        returns_clean.shape[1],
        people_clean.shape[1]
    ]
})

display(loading_summary)

,Dataset,Rows,Columns
0,Orders,51290,24
1,Returns,1079,3
2,People,24,2


## 3. Standardize Column Names

I convert all column names to lowercase snake_case so they remain consistent across Python, SQL, CSV, and Power BI workflows.

In [3]:
def standardize_column_names(dataframe):
    cleaned_dataframe = dataframe.copy()

    cleaned_dataframe.columns = (
        cleaned_dataframe.columns
        .str.strip()
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )

    return cleaned_dataframe


orders_clean = standardize_column_names(
    orders_clean
)
returns_clean = standardize_column_names(
    returns_clean
)
people_clean = standardize_column_names(
    people_clean
)

orders_column_mapping = pd.DataFrame({
    "Original Column": orders_raw.columns,
    "Clean Column": orders_clean.columns
})

display(orders_column_mapping)

,Original Column,Clean Column
0,Row ID,row_id
1,Order ID,order_id
2,Order Date,order_date
3,Ship Date,ship_date
4,Ship Mode,ship_mode
5,Customer ID,customer_id
6,Customer Name,customer_name
7,Segment,segment
8,Postal Code,postal_code
9,City,city


## 4. Trim Text Values

I remove leading and trailing whitespace from text fields while preserving spaces within names and descriptions.

In [4]:
def trim_text_values(dataframe):
    cleaned_dataframe = dataframe.copy()

    text_columns = cleaned_dataframe.select_dtypes(
        include=["object", "string"]
    ).columns

    affected_values = 0

    for column in text_columns:
        text_values = cleaned_dataframe[column].astype(
            "string"
        )

        affected_values += (
            text_values.notna()
            & (text_values != text_values.str.strip())
        ).sum()

        cleaned_dataframe[column] = (
            text_values.str.strip()
        )

    return cleaned_dataframe, int(affected_values)


orders_clean, orders_trimmed_count = (
    trim_text_values(orders_clean)
)
returns_clean, returns_trimmed_count = (
    trim_text_values(returns_clean)
)
people_clean, people_trimmed_count = (
    trim_text_values(people_clean)
)

text_cleaning_summary = pd.DataFrame({
    "Dataset": ["Orders", "Returns", "People"],
    "Trimmed Values": [
        orders_trimmed_count,
        returns_trimmed_count,
        people_trimmed_count
    ]
})

display(text_cleaning_summary)

,Dataset,Trimmed Values
0,Orders,16
1,Returns,0
2,People,0


## 5. Convert Data Types

### 5.1 Date Fields

I convert the order and shipping dates from text into validated datetime fields using the audited day-month-year format.

In [5]:
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    format="%d-%m-%y",
    errors="raise"
)

orders_clean["ship_date"] = pd.to_datetime(
    orders_clean["ship_date"],
    format="%d-%m-%y",
    errors="raise"
)

date_type_summary = pd.DataFrame({
    "Column": ["order_date", "ship_date"],
    "Data Type": [
        orders_clean["order_date"].dtype,
        orders_clean["ship_date"].dtype
    ],
    "Minimum Date": [
        orders_clean["order_date"].min(),
        orders_clean["ship_date"].min()
    ],
    "Maximum Date": [
        orders_clean["order_date"].max(),
        orders_clean["ship_date"].max()
    ]
})

display(date_type_summary)

,Column,Data Type,Minimum Date,Maximum Date
0,order_date,datetime64[us],2012-01-01,2015-12-31
1,ship_date,datetime64[us],2012-01-03,2016-01-07


### 5.2 Currency Fields

I remove currency symbols and thousands separators from Sales and Profit before converting them to numeric fields.

In [6]:
def convert_currency_to_numeric(series):
    cleaned_values = (
        series
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )

    return pd.to_numeric(
        cleaned_values,
        errors="raise"
    )


orders_clean["sales"] = (
    convert_currency_to_numeric(
        orders_clean["sales"]
    )
)

orders_clean["profit"] = (
    convert_currency_to_numeric(
        orders_clean["profit"]
    )
)

currency_type_summary = pd.DataFrame({
    "Column": ["sales", "profit"],
    "Data Type": [
        orders_clean["sales"].dtype,
        orders_clean["profit"].dtype
    ],
    "Minimum": [
        orders_clean["sales"].min(),
        orders_clean["profit"].min()
    ],
    "Maximum": [
        orders_clean["sales"].max(),
        orders_clean["profit"].max()
    ]
})

display(currency_type_summary)

,Column,Data Type,Minimum,Maximum
0,sales,Float64,0.44,22638.48
1,profit,Float64,-6599.98,8399.98


## 6. Create Analytical Keys

### 6.1 Order Key

I create an analytical Order Key from Order ID and Customer ID to prevent the 11 validated raw Order ID collisions from merging distinct customer-region order instances.

In [7]:
orders_clean["order_key"] = (
    orders_clean["order_id"]
    + "|"
    + orders_clean["customer_id"]
)

print(
    "Unique raw Order IDs:",
    orders_clean["order_id"].nunique()
)
print(
    "Unique analytical Order Keys:",
    orders_clean["order_key"].nunique()
)
print(
    "Missing analytical Order Keys:",
    orders_clean["order_key"].isna().sum()
)

Unique raw Order IDs: 25728
Unique analytical Order Keys: 25739
Missing analytical Order Keys: 0


### 6.2 Customer Key

I create one surrogate Customer Key for each distinct Customer Name based on the validated dataset-specific customer-identity rule. The original Customer ID remains available as a source account identifier.

In [8]:
customer_key_mapping = (
    orders_clean[["customer_name"]]
    .drop_duplicates()
    .sort_values("customer_name")
    .reset_index(drop=True)
)

customer_key_mapping.insert(
    0,
    "customer_key",
    np.arange(
        1,
        len(customer_key_mapping) + 1
    )
)

orders_row_count_before_merge = len(orders_clean)

orders_clean = orders_clean.merge(
    customer_key_mapping,
    on="customer_name",
    how="left",
    validate="many_to_one"
)

print(
    "Distinct Customer Names:",
    customer_key_mapping["customer_name"].nunique()
)
print(
    "Distinct Customer Keys:",
    customer_key_mapping["customer_key"].nunique()
)
print(
    "Missing Customer Keys:",
    orders_clean["customer_key"].isna().sum()
)
print(
    "Rows unchanged after merge:",
    len(orders_clean) == orders_row_count_before_merge
)

display(customer_key_mapping.head())

Distinct Customer Names: 796
Distinct Customer Keys: 796
Missing Customer Keys: 0
Rows unchanged after merge: True


,customer_key,customer_name
0,1,Aaron Bergman
1,2,Aaron Hawkins
2,3,Aaron Smayling
3,4,Adam Bellavance
4,5,Adam Hart


## 7. Feature Engineering

### 7.1 Shipping Duration

I calculate the number of days between Order Date and Ship Date for operational and shipping-performance analysis.

In [9]:
orders_clean["shipping_days"] = (
    orders_clean["ship_date"]
    - orders_clean["order_date"]
).dt.days.astype("Int64")

print(
    "Minimum Shipping Days:",
    orders_clean["shipping_days"].min()
)
print(
    "Maximum Shipping Days:",
    orders_clean["shipping_days"].max()
)
print(
    "Negative Shipping Days:",
    (
        orders_clean["shipping_days"] < 0
    ).sum()
)
print(
    "Missing Shipping Days:",
    orders_clean["shipping_days"].isna().sum()
)

Minimum Shipping Days: 0
Maximum Shipping Days: 7
Negative Shipping Days: 0
Missing Shipping Days: 0


### 7.2 Profitability Features

I create line-level profitability features while preserving Sales and Profit as the source financial fields.

In [10]:
orders_clean["sales_per_unit"] = (
    orders_clean["sales"]
    / orders_clean["quantity"]
)

orders_clean["line_profit_margin"] = (
    orders_clean["profit"]
    / orders_clean["sales"]
)

profit_conditions = [
    orders_clean["profit"]
    .lt(0)
    .fillna(False)
    .to_numpy(),

    orders_clean["profit"]
    .eq(0)
    .fillna(False)
    .to_numpy()
]

profit_labels = [
    "Loss",
    "Break-even"
]

orders_clean["profit_status"] = np.select(
    profit_conditions,
    profit_labels,
    default="Profit"
)

orders_clean["profit_status"] = (
    orders_clean["profit_status"].astype("string")
)

profitability_summary = pd.DataFrame({
    "Metric": [
        "sales_per_unit",
        "line_profit_margin"
    ],
    "Minimum": [
        orders_clean["sales_per_unit"].min(),
        orders_clean["line_profit_margin"].min()
    ],
    "Maximum": [
        orders_clean["sales_per_unit"].max(),
        orders_clean["line_profit_margin"].max()
    ]
})

display(profitability_summary)

display(
    orders_clean["profit_status"].value_counts()
)

,Metric,Minimum,Maximum
0,sales_per_unit,0.335000,3773.08000
1,line_profit_margin,-4.733637,0.50022


profit_status
Profit        38078
Loss          12540
Break-even      672
Name: count, dtype: int64[pyarrow]

## 8. Prepare and Integrate Returns

### 8.1 Standardize Return Regions

I preserve the source return-region field and create a standardized region for the validated Orders–Returns join.

In [11]:
returns_clean["standardized_region"] = (
    returns_clean["region"].replace({
        "Eastern Canada": "Canada",
        "Western Canada": "Canada"
    })
)

returns_clean["is_returned"] = True

return_region_changes = (
    returns_clean.loc[
        returns_clean["region"]
        != returns_clean["standardized_region"]
    ]
    .groupby(
        ["region", "standardized_region"]
    )
    .size()
    .reset_index(name="return_records")
)

duplicate_return_keys = returns_clean.duplicated(
    subset=["order_id", "standardized_region"]
).sum()

display(return_region_changes)

print(
    "Duplicate standardized return keys:",
    duplicate_return_keys
)

,region,standardized_region,return_records
0,Eastern Canada,Canada,5
1,Western Canada,Canada,1


Duplicate standardized return keys: 0


### 8.2 Add the Order-Level Return Flag

I join the standardized Returns lookup to Orders using Order ID and Region, then convert unmatched orders into a False return flag.

In [12]:
return_lookup = (
    returns_clean[
        [
            "order_id",
            "standardized_region",
            "is_returned"
        ]
    ]
    .drop_duplicates()
)

orders_row_count_before_return_merge = len(
    orders_clean
)

orders_clean = orders_clean.merge(
    return_lookup,
    left_on=["order_id", "region"],
    right_on=[
        "order_id",
        "standardized_region"
    ],
    how="left",
    validate="many_to_one"
)

orders_clean["is_returned"] = (
    orders_clean["is_returned"]
    .astype("boolean")
    .fillna(False)
)

orders_clean = orders_clean.drop(
    columns=["standardized_region"]
)

print(
    "Rows unchanged after return merge:",
    len(orders_clean)
    == orders_row_count_before_return_merge
)
print(
    "Returned analytical orders:",
    orders_clean.loc[
        orders_clean["is_returned"],
        "order_key"
    ].nunique()
)
print(
    "Sales lines in returned orders:",
    orders_clean["is_returned"].sum()
)
print(
    "Missing return flags:",
    orders_clean["is_returned"].isna().sum()
)

Rows unchanged after return merge: True
Returned analytical orders: 1079
Sales lines in returned orders: 2217
Missing return flags: 0


## 9. Integrate Regional Manager Mapping

I add exact regional-manager mappings and explicitly label Canadian sales as unassigned because the source datasets use incompatible Canadian region levels.

In [13]:
manager_lookup = (
    people_clean[
        ["region", "person"]
    ]
    .rename(
        columns={
            "person": "regional_manager"
        }
    )
    .drop_duplicates()
)

orders_row_count_before_manager_merge = len(
    orders_clean
)

orders_clean = orders_clean.merge(
    manager_lookup,
    on="region",
    how="left",
    validate="many_to_one"
)

unmapped_manager_rows = (
    orders_clean["regional_manager"]
    .isna()
    .sum()
)

orders_clean["manager_mapping_status"] = (
    np.where(
        orders_clean["regional_manager"].isna(),
        "Unassigned",
        "Mapped"
    )
)

orders_clean["manager_mapping_status"] = (
    orders_clean["manager_mapping_status"]
    .astype("string")
)

orders_clean["regional_manager"] = (
    orders_clean["regional_manager"]
    .fillna(
        "Unassigned - Canada mapping unavailable"
    )
    .astype("string")
)

print(
    "Rows unchanged after manager merge:",
    len(orders_clean)
    == orders_row_count_before_manager_merge
)
print(
    "Unmapped rows before labeling:",
    unmapped_manager_rows
)
print(
    "Missing managers after labeling:",
    orders_clean["regional_manager"]
    .isna()
    .sum()
)

display(
    orders_clean[
        "manager_mapping_status"
    ].value_counts()
)

Rows unchanged after manager merge: True
Unmapped rows before labeling: 384
Missing managers after labeling: 0


manager_mapping_status
Mapped        50906
Unassigned      384
Name: count, dtype: int64[pyarrow]

## 10. Create Order-Date Features

I derive reusable calendar fields from Order Date for Python, Excel, and flat-file time-series analysis. These attributes will later be modeled in DimDate for Power BI.

In [14]:
orders_clean["order_year"] = (
    orders_clean["order_date"]
    .dt.year
    .astype("Int64")
)

orders_clean["order_quarter"] = (
    "Q"
    + orders_clean["order_date"]
        .dt.quarter
        .astype("string")
)

orders_clean["order_month"] = (
    orders_clean["order_date"]
    .dt.month
    .astype("Int64")
)

orders_clean["order_month_name"] = (
    orders_clean["order_date"]
    .dt.month_name()
    .astype("string")
)

orders_clean["order_year_month"] = (
    orders_clean["order_date"]
    .dt.to_period("M")
    .astype("string")
)

time_feature_summary = pd.DataFrame({
    "Feature": [
        "order_year",
        "order_quarter",
        "order_month",
        "order_month_name",
        "order_year_month"
    ],
    "Unique Values": [
        orders_clean["order_year"].nunique(),
        orders_clean["order_quarter"].nunique(),
        orders_clean["order_month"].nunique(),
        orders_clean["order_month_name"].nunique(),
        orders_clean["order_year_month"].nunique()
    ],
    "First Value": [
        orders_clean["order_year"].min(),
        sorted(
            orders_clean["order_quarter"].unique()
        )[0],
        orders_clean["order_month"].min(),
        "January",
        orders_clean["order_year_month"].min()
    ],
    "Last Value": [
        orders_clean["order_year"].max(),
        sorted(
            orders_clean["order_quarter"].unique()
        )[-1],
        orders_clean["order_month"].max(),
        "December",
        orders_clean["order_year_month"].max()
    ]
})

display(time_feature_summary)

,Feature,Unique Values,First Value,Last Value
0,order_year,4,2012,2015
1,order_quarter,4,Q1,Q4
2,order_month,12,1,12
3,order_month_name,12,January,December
4,order_year_month,48,2012-01,2015-12


## 11. Financial Reconciliation

I confirm that data-type conversion and merging did not change the total Sales or Profit recorded in the raw dataset.

In [15]:
raw_sales_total = (
    convert_currency_to_numeric(
        orders_raw["Sales"]
        .astype("string")
        .str.strip()
    )
    .sum()
)

raw_profit_total = (
    convert_currency_to_numeric(
        orders_raw["Profit"]
        .astype("string")
        .str.strip()
    )
    .sum()
)

financial_reconciliation = pd.DataFrame({
    "Metric": ["Sales", "Profit"],
    "Raw Total": [
        raw_sales_total,
        raw_profit_total
    ],
    "Clean Total": [
        orders_clean["sales"].sum(),
        orders_clean["profit"].sum()
    ]
})

financial_reconciliation["Difference"] = (
    financial_reconciliation["Clean Total"]
    - financial_reconciliation["Raw Total"]
)

financial_reconciliation["Passed"] = np.isclose(
    financial_reconciliation["Raw Total"],
    financial_reconciliation["Clean Total"]
)

display(financial_reconciliation)

print(
    "Financial reconciliation passed:",
    financial_reconciliation["Passed"].all()
)

,Metric,Raw Total,Clean Total,Difference,Passed
0,Sales,12642507.25,12642507.25,0.0,True
1,Profit,1467456.67,1467456.67,0.0,True


Financial reconciliation passed: True


## 12. Final Cleaning Validation

I run final control checks to confirm that cleaning preserved source records, financial totals, key integrity, and valid business ranges.

In [16]:
unexpected_missing_values = (
    orders_clean
    .drop(columns=["postal_code"])
    .isna()
    .sum()
    .sum()
)
required_clean_columns = [
    "row_id",
    "order_id",
    "order_key",
    "customer_id",
    "customer_key",
    "product_id",
    "order_date",
    "ship_date",
    "sales",
    "profit",
    "shipping_days",
    "sales_per_unit",
    "line_profit_margin",
    "profit_status",
    "is_returned",
    "regional_manager",
    "manager_mapping_status",
    "order_year",
    "order_quarter",
    "order_month",
    "order_month_name",
    "order_year_month"
]

missing_required_columns = [
    column
    for column in required_clean_columns
    if column not in orders_clean.columns
]

validation_checks = [
    {
        "Validation Check": "Orders row count preserved",
        "Passed": len(orders_clean) == len(orders_raw)
    },
    {
        "Validation Check": "Returns row count preserved",
        "Passed": len(returns_clean) == len(returns_raw)
    },
    {
        "Validation Check": "People row count preserved",
        "Passed": len(people_clean) == len(people_raw)
    },
    {
        "Validation Check": "No required cleaned columns missing",
        "Passed": len(missing_required_columns) == 0
    },
    {
        "Validation Check": "Row ID remains unique",
        "Passed": orders_clean["row_id"].is_unique
    },
    {
        "Validation Check": "No missing analytical Order Keys",
        "Passed": orders_clean["order_key"].notna().all()
    },
    {
        "Validation Check": "No missing Customer Keys",
        "Passed": orders_clean["customer_key"].notna().all()
    },
    {
        "Validation Check": "Quantity remains positive",
        "Passed": orders_clean["quantity"].gt(0).all()
    },
    {
        "Validation Check": "Sales remains positive",
        "Passed": orders_clean["sales"].gt(0).all()
    },
    {
        "Validation Check": "Discount remains between 0 and 1",
        "Passed": orders_clean["discount"].between(0, 1).all()
    },
    {
        "Validation Check": "Shipping duration is non-negative",
        "Passed": orders_clean["shipping_days"].ge(0).all()
    },
    {
        "Validation Check": "No missing return flags",
        "Passed": orders_clean["is_returned"].notna().all()
    },
    {
        "Validation Check": "Returned-order count reconciles",
        "Passed": (
            orders_clean.loc[
                orders_clean["is_returned"],
                "order_key"
            ].nunique()
            == len(returns_clean)
        )
    },
    {
        "Validation Check": "No missing manager labels",
        "Passed": orders_clean["regional_manager"].notna().all()
    },
    {
        "Validation Check": "Sales total reconciles",
        "Passed": np.isclose(
            raw_sales_total,
            orders_clean["sales"].sum()
        )
    },
    {
        "Validation Check": "Profit total reconciles",
        "Passed": np.isclose(
            raw_profit_total,
            orders_clean["profit"].sum()
        )
    }
]
validation_checks.extend([
    {
        "Validation Check": "No exact duplicate rows",
        "Passed": orders_clean.duplicated().sum() == 0
    },
    {
        "Validation Check": "No unexpected missing values",
        "Passed": unexpected_missing_values == 0
    },
    {
        "Validation Check": "All 48 order months available",
        "Passed": (
            orders_clean["order_year_month"].nunique()
            == 48
        )
    }
])
validation_summary = pd.DataFrame(
    validation_checks
)

display(validation_summary)

print(
    "Missing required columns:",
    missing_required_columns
)
print(
    "All cleaning validations passed:",
    validation_summary["Passed"].all()
)

,Validation Check,Passed
0,Orders row count preserved,True
1,Returns row count preserved,True
2,People row count preserved,True
3,No required cleaned columns missing,True
4,Row ID remains unique,True
5,No missing analytical Order Keys,True
6,No missing Customer Keys,True
7,Quantity remains positive,True
8,Sales remains positive,True
9,Discount remains between 0 and 1,True


Missing required columns: []
All cleaning validations passed: True


## 13. Prepare Processed Outputs

I arrange the cleaned sales fields into a logical business order and prepare supporting lookup files without removing any validated columns.

In [17]:
orders_output_columns = [
    # Sales-line and order identifiers
    "row_id",
    "order_key",
    "order_id",

    # Dates and time features
    "order_date",
    "ship_date",
    "shipping_days",
    "order_year",
    "order_quarter",
    "order_month",
    "order_month_name",
    "order_year_month",

    # Order attributes
    "ship_mode",
    "order_priority",

    # Customer attributes
    "customer_key",
    "customer_id",
    "customer_name",
    "segment",

    # Geography and manager attributes
    "postal_code",
    "city",
    "state",
    "country",
    "region",
    "market",
    "regional_manager",
    "manager_mapping_status",

    # Product attributes
    "product_id",
    "product_name",
    "category",
    "sub_category",

    # Financial and operational metrics
    "quantity",
    "sales",
    "sales_per_unit",
    "discount",
    "profit",
    "line_profit_margin",
    "profit_status",
    "shipping_cost",
    "is_returned"
]

missing_output_columns = sorted(
    set(orders_output_columns)
    - set(orders_clean.columns)
)

extra_clean_columns = sorted(
    set(orders_clean.columns)
    - set(orders_output_columns)
)

if missing_output_columns or extra_clean_columns:
    raise ValueError(
        "Output-column mismatch detected. "
        f"Missing: {missing_output_columns}; "
        f"Extra: {extra_clean_columns}"
    )

orders_output = orders_clean[
    orders_output_columns
].copy()

returns_output = (
    returns_clean[
        [
            "order_id",
            "returned",
            "region",
            "standardized_region",
            "is_returned"
        ]
    ]
    .rename(
        columns={"region": "return_region"}
    )
    .copy()
)

customer_mapping_output = (
    customer_key_mapping[
        ["customer_key", "customer_name"]
    ]
    .copy()
)

manager_mapping_output = (
    manager_lookup[
        ["region", "regional_manager"]
    ]
    .copy()
)

processed_output_summary = pd.DataFrame({
    "Output": [
        "orders_cleaned",
        "returns_cleaned",
        "customer_key_mapping",
        "regional_manager_mapping"
    ],
    "Rows": [
        len(orders_output),
        len(returns_output),
        len(customer_mapping_output),
        len(manager_mapping_output)
    ],
    "Columns": [
        orders_output.shape[1],
        returns_output.shape[1],
        customer_mapping_output.shape[1],
        manager_mapping_output.shape[1]
    ]
})

display(processed_output_summary)

,Output,Rows,Columns
0,orders_cleaned,51290,38
1,returns_cleaned,1079,5
2,customer_key_mapping,796,2
3,regional_manager_mapping,24,2


## 14. Export Processed CSV Files

I export the validated analysis-ready sales data and supporting lookup tables to the processed data folder using portable UTF-8 CSV files.

In [18]:
processed_exports = {
    "orders_cleaned.csv": orders_output,
    "returns_cleaned.csv": returns_output,
    "customer_key_mapping.csv": customer_mapping_output,
    "regional_manager_mapping.csv": manager_mapping_output
}

export_results = []

for file_name, dataframe in processed_exports.items():
    output_path = (
        processed_data_folder / file_name
    )

    dataframe.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d",
        lineterminator="\n"
    )

    export_results.append({
        "File": file_name,
        "Rows": len(dataframe),
        "Columns": dataframe.shape[1],
        "File Exists": output_path.exists(),
        "Size MB": round(
            output_path.stat().st_size
            / (1024 ** 2),
            2
        )
    })

export_summary = pd.DataFrame(
    export_results
)

display(export_summary)

,File,Rows,Columns,File Exists,Size MB
0,orders_cleaned.csv,51290,38,True,18.70
1,returns_cleaned.csv,1079,5,True,0.06
2,customer_key_mapping.csv,796,2,True,0.01
3,regional_manager_mapping.csv,24,2,True,0.00


## 15. Export Round-Trip Validation

I reload the exported CSV files and compare them with the cleaned DataFrames. This confirms that row counts, identifiers, dates, financial totals, missing values, and analytical flags remain consistent after export.

In [19]:
# Reload the exported processed files

orders_export_check = pd.read_csv(
    processed_data_folder / "orders_cleaned.csv",
    encoding="utf-8-sig",
    dtype={"postal_code": "string"},
    parse_dates=["order_date", "ship_date"]
)

returns_export_check = pd.read_csv(
    processed_data_folder / "returns_cleaned.csv",
    encoding="utf-8-sig"
)

customer_export_check = pd.read_csv(
    processed_data_folder / "customer_key_mapping.csv",
    encoding="utf-8-sig"
)

manager_export_check = pd.read_csv(
    processed_data_folder / "regional_manager_mapping.csv",
    encoding="utf-8-sig"
)


# Validate the reloaded files

round_trip_checks = pd.DataFrame({
    "Validation Check": [
        "Orders export shape preserved",
        "Returns export shape preserved",
        "Customer mapping shape preserved",
        "Manager mapping shape preserved",
        "Row ID sequence preserved",
        "Order Date reloaded as datetime",
        "Ship Date reloaded as datetime",
        "Analytical Order Key count preserved",
        "Customer Key count preserved",
        "Returned-order count preserved",
        "Postal Code missing values preserved",
        "Sales total preserved after export",
        "Profit total preserved after export"
    ],
    "Passed": [
        orders_export_check.shape == orders_output.shape,
        returns_export_check.shape == returns_output.shape,
        customer_export_check.shape == customer_mapping_output.shape,
        manager_export_check.shape == manager_mapping_output.shape,
        orders_export_check["row_id"].equals(
            orders_output["row_id"].reset_index(drop=True)
        ),
        pd.api.types.is_datetime64_any_dtype(
            orders_export_check["order_date"]
        ),
        pd.api.types.is_datetime64_any_dtype(
            orders_export_check["ship_date"]
        ),
        orders_export_check["order_key"].nunique()
        == orders_output["order_key"].nunique(),
        orders_export_check["customer_key"].nunique()
        == orders_output["customer_key"].nunique(),
        orders_export_check.loc[
            orders_export_check["is_returned"], "order_key"
        ].nunique()
        == orders_output.loc[
            orders_output["is_returned"], "order_key"
        ].nunique(),
        orders_export_check["postal_code"].isna().sum()
        == orders_output["postal_code"].isna().sum(),
        np.isclose(
            orders_export_check["sales"].sum(),
            orders_output["sales"].sum()
        ),
        np.isclose(
            orders_export_check["profit"].sum(),
            orders_output["profit"].sum()
        )
    ]
})

display(round_trip_checks)

print(
    "All exported files passed round-trip validation:",
    round_trip_checks["Passed"].all()
)

,Validation Check,Passed
0,Orders export shape preserved,True
1,Returns export shape preserved,True
2,Customer mapping shape preserved,True
3,Manager mapping shape preserved,True
4,Row ID sequence preserved,True
5,Order Date reloaded as datetime,True
6,Ship Date reloaded as datetime,True
7,Analytical Order Key count preserved,True
8,Customer Key count preserved,True
9,Returned-order count preserved,True


All exported files passed round-trip validation: True


## 16. Cleaning and Feature Engineering Summary

I transformed the raw Global Superstore data into analysis-ready datasets while preserving the original financial values and transaction records.

### Key Data Decisions

- Standardized column names and removed unwanted whitespace from text fields.
- Converted order dates, shipping dates, sales, and profit into appropriate analytical data types.
- Created a validated analytical order key to handle 11 duplicated raw Order IDs.
- Created a customer surrogate key for 796 distinct customers while retaining the original Customer ID.
- Standardized Canadian return regions to match all 1,079 returned orders correctly.
- Preserved all valid outliers and 12,540 loss-making sales lines because they represent genuine business activity.
- Labeled 384 Canadian sales lines as having unavailable manager mapping instead of assigning an unsupported manager.
- Created shipping, profitability, customer, return, and time-based analytical features.
- Reconciled the cleaned sales and profit totals with the raw dataset.
- Verified the exported CSV files by reloading and validating them.

### Final Processed Outputs

| Output File | Purpose |
|---|---|
| `orders_cleaned.csv` | Main analysis-ready sales dataset |
| `returns_cleaned.csv` | Standardized order-return information |
| `customer_key_mapping.csv` | Customer surrogate-key mapping |
| `regional_manager_mapping.csv` | Regional manager reference table |

All cleaning, reconciliation, and export validations passed successfully.